<a href="https://colab.research.google.com/github/tbruffy/econometrics-ps2/blob/main/notebooks/ps2_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import pandas as pd
from google.colab import files

# Select ps1small.csv from your computer
uploaded = files.upload()

df = pd.read_csv("ps1small.csv")

print("Dataset dimensions:", df.shape)
print(df.head())
print("\nMissing values:")
print(df.isna().sum())

Saving ps1small.csv to ps1small (1).csv
Dataset dimensions: (898, 3)
    wage  education  age
0  30000         17   30
1  16000         12   27
2  43500         17   28
3  18000         12   30
4  25000         17   27

Missing values:
wage         0
education    0
age          0
dtype: int64


In [3]:
# Logarithms require strictly positive wages
assert (df["wage"] > 0).all()

df["lwage"] = np.log(df["wage"])

df.head()

,wage,education,age,lwage
0,30000,17,30,10.308953
1,16000,12,27,9.680344
2,43500,17,28,10.680516
3,18000,12,30,9.798127
4,25000,17,27,10.126631


In [4]:
# Create an empty table with one row per person
D = pd.DataFrame(index=df.index)

# Create one dummy for each education–age combination
for education in range(12, 18):
    for age in range(26, 31):
        name = f"d{education}{age}"

        D[name] = (
            (df["education"] == education)
            & (df["age"] == age)
        ).astype(int)

print("Dummy matrix dimensions:", D.shape)

# Check that each person belongs to exactly one group
assert (D.sum(axis=1) == 1).all()

D.head()

Dummy matrix dimensions: (898, 30)


,d1226,d1227,d1228,d1229,d1230,d1326,d1327,d1328,d1329,d1330,...,d1626,d1627,d1628,d1629,d1630,d1726,d1727,d1728,d1729,d1730
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
1,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
3,0,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0


In [5]:
import statsmodels.api as sm

# Regress log wage on all 30 dummies, without an intercept
ols_unrestricted = sm.OLS(df["lwage"], D).fit()

# Collect the estimates, standard errors, and group sizes
estimates = pd.DataFrame({
    "alpha_hat": ols_unrestricted.params,
    "standard_error": ols_unrestricted.bse,
    "group_size": D.sum()
})

print(estimates.round(4).to_string())

       alpha_hat  standard_error  group_size
d1226     9.4513          0.0908          85
d1227     9.3004          0.0850          97
d1228     9.5379          0.0833         101
d1229     9.3858          0.0854          96
d1230     9.6283          0.0877          91
d1326     9.6121          0.1920          19
d1327     9.7559          0.1920          19
d1328     9.3782          0.1745          23
d1329     9.8200          0.2237          14
d1330     9.4465          0.2030          17
d1426     9.5807          0.2237          14
d1427     9.7944          0.1708          24
d1428     9.6422          0.2161          15
d1429     9.7122          0.1745          23
d1430     9.6454          0.1708          24
d1526     9.5346          0.2416          12
d1527     9.3856          0.2321          13
d1528     9.5732          0.2647          10
d1529     9.5404          0.2524          11
d1530     9.9195          0.2416          12
d1626    10.0705          0.1674          25
d1627     

In [8]:
estimates.to_latex(
    "ps1a_results.tex",
    float_format="%.4f",
    header=[
        r"$\widehat{{\alpha}}$",
        "Standard error",
        "Group size"
    ],
    column_format="lrrr",
    escape=False
)

files.download("ps1a_results.tex")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
# Start with education and age
X_restricted = df[["education", "age"]].copy()

# Add their interaction
X_restricted["education_age"] = (
    df["education"] * df["age"]
)

# This model requires an intercept
X_restricted = sm.add_constant(X_restricted)

# Estimate the restricted regression
ols_restricted = sm.OLS(
    df["lwage"], X_restricted
).fit()

print(ols_restricted.params)

const            6.107505
education        0.197808
age              0.064883
education_age   -0.002520
dtype: float64


In [10]:
from scipy import stats

ssr_u = ols_unrestricted.ssr
ssr_r = ols_restricted.ssr

num_restrictions = 26
df_residual = int(ols_unrestricted.df_resid)  # 868

F_stat = (
    (ssr_r - ssr_u) / num_restrictions
) / (ssr_u / df_residual)

p_value = stats.f.sf(
    F_stat, num_restrictions, df_residual
)

print(f"Unrestricted SSR: {ssr_u:.4f}")
print(f"Restricted SSR:   {ssr_r:.4f}")
print(f"F(26, 868):       {F_stat:.4f}")
print(f"p-value:          {p_value:.4f}")

Unrestricted SSR: 608.0274
Restricted SSR:   624.7819
F(26, 868):       0.9199
p-value:          0.5806


## Problem 3(e): Comparing log-wage variances

In [11]:
import numpy as np
import pandas as pd
from scipy import stats
from google.colab import files

# Choose wage.xlsx from Downloads
uploaded_wage = files.upload()

wages_df = pd.read_excel("wage.xlsx")

# Check the variables needed for this analysis
assert not wages_df[["male", "wage"]].isna().any().any()
assert wages_df["male"].isin([0, 1]).all()
assert (wages_df["wage"] > 0).all()

wages_df["lwage"] = np.log(wages_df["wage"])

print("Total observations:", len(wages_df))
print("Female observations:", (wages_df["male"] == 0).sum())
print("Male observations:", (wages_df["male"] == 1).sum())

Saving wage.xlsx to wage.xlsx
Total observations: 3296
Female observations: 1569
Male observations: 1727


In [12]:
# Group 1: females; group 2: males
female_lwage = wages_df.loc[
    wages_df["male"] == 0, "lwage"
].to_numpy()

male_lwage = wages_df.loc[
    wages_df["male"] == 1, "lwage"
].to_numpy()

n_female = len(female_lwage)
n_male = len(male_lwage)

# Sample variances: divide by n - 1
var_female = np.var(female_lwage, ddof=1)
var_male = np.var(male_lwage, ddof=1)

T = np.sqrt(
    n_female * n_male / (n_female + n_male)
) * (np.log(var_female) - np.log(var_male))

# Center observations around their own group means
female_centered = female_lwage - female_lwage.mean()
male_centered = male_lwage - male_lwage.mean()

sum_squared = (
    np.sum(female_centered**2)
    + np.sum(male_centered**2)
)

sum_fourth = (
    np.sum(female_centered**4)
    + np.sum(male_centered**4)
)

# The pooled kurtosis estimator specified in the question
kappa_hat = (
    (n_female + n_male) * sum_fourth
    / sum_squared**2
)

print(f"Female sample variance: {var_female:.6f}")
print(f"Male sample variance:   {var_male:.6f}")
print(f"T statistic:            {T:.6f}")
print(f"Estimated kurtosis:     {kappa_hat:.6f}")

Female sample variance: 0.397877
Male sample variance:   0.366413
T statistic:            2.362096
Estimated kurtosis:     7.378121


In [13]:
# Kurtosis-adjusted asymptotic test
Z_robust = T / np.sqrt(kappa_hat - 1)
p_robust = stats.norm.sf(Z_robust)

# Standard normal-theory F-test
F_stat_wage = var_female / var_male

p_normal_F = stats.f.sf(
    F_stat_wage,
    n_female - 1,
    n_male - 1
)

test_results = pd.DataFrame({
    "Test": [
        "Kurtosis-adjusted",
        "Normal-theory F"
    ],
    "Statistic": [
        Z_robust,
        F_stat_wage
    ],
    "p-value": [
        p_robust,
        p_normal_F
    ],
    "Decision at 5%": [
        "Reject H0" if p_robust < 0.05 else "Do not reject H0",
        "Reject H0" if p_normal_F < 0.05 else "Do not reject H0"
    ]
}).set_index("Test")

print(test_results.round(6).to_string())

                   Statistic   p-value    Decision at 5%
Test                                                    
Kurtosis-adjusted   0.935300  0.174817  Do not reject H0
Normal-theory F     1.085871  0.047347         Reject H0


In [14]:
test_results.to_latex(
    "ps3e_results.tex",
    float_format="%.4f",
    column_format="lrrl",
    escape=True
)

files.download("ps3e_results.tex")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>